# 6.3.3 Retrieval-Augmented Generation (RAG)

Build a TF-IDF document store and perform cosine similarity retrieval for top-k context assembly.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
import math
from pathlib import Path

Path('output').mkdir(exist_ok=True)

CORPUS = [
    'Machine learning is a subset of artificial intelligence.',
    'Deep learning uses neural networks with many layers.',
    'Transformers use attention mechanisms for sequence modelling.',
    'RAG combines retrieval with language model generation.',
    'Vector databases store embeddings for similarity search.',
    'TF-IDF weights terms by term and document frequency.',
]

def tokenize(t): return t.lower().split()

N = len(CORPUS)
tok = [tokenize(d) for d in CORPUS]
df = Counter(w for doc in tok for w in set(doc))
vocab = sorted(df)
v2i = {w: i for i, w in enumerate(vocab)}

def tfidf_vec(tokens):
    tf = Counter(tokens)
    v = np.zeros(len(vocab))
    for w, c in tf.items():
        if w in v2i:
            v[v2i[w]] = (c/len(tokens)) * math.log((N+1)/(df[w]+1))
    norm = np.linalg.norm(v)
    return v / (norm + 1e-10)

vecs = np.array([tfidf_vec(t) for t in tok])
print(f'TF-IDF matrix: {vecs.shape}  vocab={len(vocab)}')

In [ ]:
query = 'how do transformers work'
qvec = tfidf_vec(tokenize(query))
scores = vecs @ qvec
top_k = np.argsort(scores)[::-1][:3]

print(f'Query: "{query}"')
for i in top_k:
    print(f'  [{scores[i]:.3f}] {CORPUS[i]}')

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['tomato' if i in top_k else 'steelblue' for i in range(N)]
ax.barh(range(N), scores, color=colors)
ax.set(yticks=range(N), yticklabels=[f'D{i}: {CORPUS[i][:45]}...' for i in range(N)],
       xlabel='Cosine Similarity', title=f'RAG Retrieval: "{query}"')
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('output/rag_retrieval.png')
print('Saved rag_retrieval.png')